# Day 7: Tools

LangChain in 10 Days · KGP Talkie · Laxmi Kant Tiwari · kgptalkie.com

**Today:** hand the model a Python function it can ask you to run. Complete the round trip by hand, learn why the docstring is the most important line you write, then add a second tool that reaches the live web.

In [1]:
import time

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv()

model = ChatOpenRouter(model="nvidia/nemotron-3.5-lightning:free", temperature=0)

## 1. Ask it something it cannot know

In [2]:
model.invoke("What is the US dollar to Indian rupee rate right now? One sentence.").text

'I don’t have access to real-time financial data, so I can’t provide the exact current USD to INR rate, but you can check live updates on financial platforms like XE, OANDA, or your bank’s website.'

In [3]:
model.invoke("How many kg of checked baggage does a Zephyr Airlines economy fare include?").text

'I couldn\'t find publicly available route or operational data for **Zephyr Airlines** in major airline databases. It may be a regional, charter, or recently established carrier, and baggage policies can vary significantly even within the same airline depending on route, country, and fare rules.\n\n**How economy checked baggage is typically structured:**\n- **Weight-based (common in Europe, Asia, Latin America):** 1–2 bags at 20–23 kg each (≈44–50 lb), often regardless of piece count\n- **Piece-based (common on many US carriers):** 1 free checked bag (usually 23 kg / 50 lb) for Main Cabin/Standard fares; Basic Economy may exclude checked bags or charge extra\n- **Low-cost carriers:** Some charge for every bag; Southwest (U.S.) famously includes 2 free bags at 50 lb each\n- **International vs. domestic:** Long-haul economy often includes at least 1 bag (23 kg), sometimes 2, while domestic economy may be more restrictive or à la carte\n\n**To get the exact policy for Zephyr Airlines:**\n

Two honest refusals, two different reasons: **live data** (trained and frozen, today's rate was never in it), and **your data** (Zephyr Airlines is made up, no public training would help).

Tools aren't about making the model cleverer, they connect it to things true right now.

## 2. Why give a model a calculator at all

Models supposedly can't do arithmetic. Check rather than believe it.

In [4]:
394857 * 682913

269652978441

Compare those two. If they match, the usual argument for giving a model a calculator just failed in front of you.

So why give it one anyway?

- It **predicted** those digits rather than calculating them, and nothing in the answer tells you which.
- It spent tokens and time doing it.
- A tool call is something you can log, test, and show an auditor. A lucky guess isn't.

## 3. A tool is a Python function

Write the function. Add `@tool`. That is all.

In [8]:
from langchain.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together and return the exact result."""
    return a * b

In [9]:
multiply

StructuredTool(name='multiply', description='Multiply two numbers together and return the exact result.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x0000029E78CA2480>)

In [12]:
# multiply(394857,682913)
# multiply.invoke

## 4. What the model receives: name, description, schema

Not your code. Three things about it.

In [13]:
multiply.name

'multiply'

In [14]:
multiply.description

'Multiply two numbers together and return the exact result.'

In [15]:
multiply.args

{'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}

The **docstring became the description**, how the model decides whether this tool fits a question.

Argument types come from type hints: write `a: float` and the model is told to send a number.

Here's the whole package as the model receives it.

In [16]:
multiply.args_schema.model_json_schema()

{'description': 'Multiply two numbers together and return the exact result.',
 'properties': {'a': {'title': 'A', 'type': 'number'},
  'b': {'title': 'B', 'type': 'number'}},
 'required': ['a', 'b'],
 'title': 'multiply',
 'type': 'object'}

That JSON is the entire contract. The model never sees your function body, everything it knows has to be in the name, the docstring, and the type hints.

## 5. Offer the tool with bind_tools

In [17]:
# time.sleep(4)

with_tools = model.bind_tools([multiply])

reply = with_tools.invoke("What is 394857 times 682913?")

reply.tool_calls

[{'name': 'multiply',
  'args': {'b': 682913, 'a': 394857},
  'id': 'call-1d50b103-0535-49b1-944f-5f78be8013ec',
  'type': 'tool_call'}]

There is no answer yet.

In [19]:
reply

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to compute 394857 * 682913. Use multiply function.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'We need to compute 394857 * 682913. Use multiply function.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3.5-lightning:free', 'id': 'gen-1786878123-YBeCsaVPqb2QjD8TScGb', 'created': 1786878123, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a00a3c-5047-7d13-8b6a-e8e7bd37aea0-0', tool_calls=[{'name': 'multiply', 'args': {'b': 682913, 'a': 394857}, 'id': 'call-1d50b103-0535-49b1-944f-5f78be8013ec', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 312, 'output_tokens': 68, 'total_tokens': 380, 'input_token_details': {'ca

Empty. The model didn't multiply anything, it **asked you to**, reading the numbers out of the question and filling in the arguments itself.

## 6. Your code runs the tool, not the model

In [20]:
call = reply.tool_calls[0]

call

{'name': 'multiply',
 'args': {'b': 682913, 'a': 394857},
 'id': 'call-1d50b103-0535-49b1-944f-5f78be8013ec',
 'type': 'tool_call'}

In [22]:
multiply.invoke(call["args"])

269652978441.0

The model never runs your code, it writes a request and **your program decides whether to honour it**.

That separation is why this is safe enough for customers. A tool call is a suggestion arriving over the network, treat it as carefully as any other outside input.

## 7. Send the result back as a ToolMessage

The model still has to turn the number into a sentence, and needs the result sent back. Pass the whole tool call, not just the arguments, and you get a `ToolMessage` ready to send.

In [26]:
tool_result = multiply.invoke(call)

tool_result

ToolMessage(content='269652978441.0', name='multiply', tool_call_id='call-1d50b103-0535-49b1-944f-5f78be8013ec')

`tool_call_id` matches the `id` on the model's call, how a reply gets matched to its request when several tools run at once.

Now build the conversation: what you asked, what it decided, what your code returned.

In [31]:
time.sleep(4)

from langchain.messages import HumanMessage

conversation = [
    HumanMessage("What is 394857 times 682913?"),
    reply,
    tool_result,
]

final = with_tools.invoke(conversation)

print(final.text)

The product of 394 857 × 682 913 is **269,652,978,441**.


In [29]:
conversation

[HumanMessage(content='What is 394857 times 682913?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to compute 394857 * 682913. Use multiply function.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'We need to compute 394857 * 682913. Use multiply function.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3.5-lightning:free', 'id': 'gen-1786878123-YBeCsaVPqb2QjD8TScGb', 'created': 1786878123, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a00a3c-5047-7d13-8b6a-e8e7bd37aea0-0', tool_calls=[{'name': 'multiply', 'args': {'b': 682913, 'a': 394857}, 'id': 'call-1d50b103-0535-49b1-944f-5f78be8013ec', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

The complete loop:

1. Send the question and the list of tools.
2. The model replies with a **tool call** instead of an answer.
3. Your code runs the function.
4. Send the result back as a `ToolMessage`.
5. The model writes the sentence.

Two model calls, one function call. Every agent framework is a wrapper around these five steps, and tomorrow you'll let one do it for you.

## 8. A real tool: web search

Section 1 opened with a refusal: the model cannot know today's exchange rate. `multiply` never had that problem, arithmetic does not go stale. A web search tool handles the questions that do.

Ollama runs a free web search API. Create a free key at https://ollama.com/settings/keys, put it in `.env` as `OLLAMA_API_KEY`, and the `load_dotenv()` at the top of this notebook picks it up.

In [32]:
import ollama

@tool
def web_search(query: str) -> str:
    """Search the live web and return the top results. Use for questions about current events, prices or rates."""
    results = ollama.web_search(query, max_results=3).results
    parts = []
    for r in results:
        parts.append(r.title + " (" + r.url + ")\n" + r.content[:300])
    return "\n\n".join(parts)

In [33]:
web_search

StructuredTool(name='web_search', description='Search the live web and return the top results. Use for questions about current events, prices or rates.', args_schema=<class 'langchain_core.utils.pydantic.web_search'>, func=<function web_search at 0x0000029E7F664C20>)

The function calls Ollama's search service and flattens the top three hits into one string, because a tool result has to travel back to the model as text. Run it directly first.

In [37]:
# print(web_search.invoke({"query": "USD to INR exchange rate today"}))

Live results, fetched just now. Hand it to the model and rerun the exact question it refused in section 1.

In [ ]:
# time.sleep(4)

searching = model.bind_tools([web_search])

question = "What is the US dollar to Indian rupee rate right now? One sentence."

reply = searching.invoke(question)

In [40]:
reply.tool_calls

[{'name': 'web_search',
  'args': {'query': 'USD to INR exchange rate now'},
  'id': 'call-838f99fa-c42d-4640-a39f-3f5a0ac5cc46',
  'type': 'tool_call'}]

In [41]:
# time.sleep(4)

conversation = [
    HumanMessage(question),
    reply,
    web_search.invoke(reply.tool_calls[0]),
]

response = searching.invoke(conversation)

In [44]:
print(response.content)

The current USD → INR exchange rate is approximately 95.42 Indian rupees per US dollar.


The same five steps as section 7, only the function changed. Section 1's refusal is now an answer with a source behind it.

Two tools, two gaps closed: `multiply` removes the guess, `web_search` removes the staleness. The loop stayed identical, and tomorrow `create_agent` runs it for you, deciding on its own how many times to go around.